In [97]:
from PIL import Image
import torch
import torchvision.models as models
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import glob
import os

Lpad the data we are gonna use 

In [98]:
DATA_DIR = r"C:\Users\chrys\Downloads\archive\brisc2025"
#training for classification for tumor images 
DATA_ctp= r"C:\Users\chrys\Downloads\archive\brisc2025\classification_task\train\pituitary"
DATA_ctm= r"C:\Users\chrys\Downloads\archive\brisc2025\classification_task\train\meningioma"
DATA_ctg= r"C:\Users\chrys\Downloads\archive\brisc2025\classification_task\train\glioma"
DATA_ctnt= r"C:\Users\chrys\Downloads\archive\brisc2025\classification_task\train\no_tumor"
# training data for segmentation task
DATA_stm= r"C:\Users\chrys\Downloads\archive\brisc2025\segmentation_task\train\masks"
DATA_sti= r"C:\Users\chrys\Downloads\archive\brisc2025\segmentation_task\train\images"


# mapping plane code  and tumors for MRI images to their corresponding labels 
PLANE_CODE_TO_LABEL = {"ax":"axial", "co":"coronal", "sa":"sagittal"}
TUMOR_TYPE_TO_LABEL = {"no": 0, "gl": 1, "me": 2, "pi": 3}


Define the dataset 

In [99]:

class BRISC2025Dataset(Dataset):

    def __init__(self, img_dir, mask_dir, exclude_plane=None, transform=None, mask_transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.mask_transform = mask_transform
        self.samples=[]
       
        # gathering all image and mask file paths
        all_img_paths=glob.glob(os.path.join(img_dir, "*.png"))+ glob.glob(os.path.join(img_dir, "*.jpg"))
        
        for img_path in all_img_paths:  
            img_name=os.path.basename(img_path)
            name_without_ext = os.path.splitext(img_name)[0]

            mask_path=os.path.join(mask_dir, name_without_ext + ".png")
           
            # plane and tumor code extraction from the image name
            name_without_ext=os.path.splitext(img_name)[0]
            parts=name_without_ext.split("_")
            tumor_code=parts[-3]
            plane_code=parts[-2]
            #Exclude planbe we want to expirement at 
            if exclude_plane and plane_code==exclude_plane:
                continue
            tumor_label=TUMOR_TYPE_TO_LABEL.get(tumor_code, 0)  # Default to 0 if tumor code not found
            # dictionary for smaple if maks exit 
            if os.path.exists(mask_path):
                self.samples.append({
                    "img_path": img_path,
                    "mask_path": mask_path,
                    "tumor_label": tumor_label,
                    "plane_code": plane_code
                })
            else:
                print(f"Warning: Mask not found for image {img_name}. Skipping this sample.")
    def __len__(self):
            return len(self.samples)
    def __getitem__(self, idx):
            sample=self.samples[idx]
            img=Image.open(sample["img_path"]).convert("RGB")
            mask=Image.open(sample["mask_path"]).convert("L")  # Assuming mask is single channel
            
            if self.transform:
                img=self.transform(img)
            if self.mask_transform:
                mask=self.mask_transform(mask)
            
            return {
                "image": img,
                "mask": mask,
                "tumor_label": sample["tumor_label"],
                "plane_code": sample["plane_code"]
            }
           

Data tranformations and Dataloaders 

In [100]:
# image tranfrormation for training and validation 
image_transform=transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225])
])
mask_transform=transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.ToTensor()
])
# create the dataset for training and validation for experimentation
train_dataset=BRISC2025Dataset(
    img_dir=DATA_sti,
    mask_dir=DATA_stm,      
    exclude_plane="co",  # Exclude coronal plane for training
    transform=image_transform,  
    mask_transform=mask_transform
)   
train_loader=DataLoader(train_dataset, batch_size=16, shuffle=True) 
#veryfies the dataset and loader
print(f"Number of training samples: {len(train_dataset)}")


Number of training samples: 2567


Resnet -18 and U-Net decoder  creation 

In [101]:
class MultiTaskResNetUNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        # Load pre-trained ResNet-18 model
        resnet=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # encoder part of the model, shared 
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool

        self.layer1=resnet.layer1
        self.layer2=resnet.layer2          
        self.layer3=resnet.layer3
        self.layer4=resnet.layer4
        # classification branch
        self.avgpool=resnet.avgpool
        self.fc=nn.Linear(resnet.fc.in_features, num_classes)
        #U-Net decoder part for segmentation
        self.upconv4=nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4=self._conv_block(512, 256)
        self.upconv3=nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3=self._conv_block(256, 128)    
        self.upconv2=nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2=self._conv_block(128, 64)
        self.upconv1=nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1=self._conv_block(128, 64)
        self.final_upsample = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.final_conv=nn.Conv2d(64, 1, kernel_size=1)  #  binary segmentation
        # cuda state for the model
        self.cls_stream=None
        self.seg_stream=None

    def _conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def _classification_branch(self, x5):
        return self.fc(self.avgpool(x5).flatten(1))
    def _decoder_branch(self, x5, x4, x3, x2, x0):
        s=self.dec4(torch.cat((self.upconv4(x5), x4), dim=1))
        s=self.dec3(torch.cat((self.upconv3(s), x3), dim=1))
        s=self.dec2(torch.cat((self.upconv2(s), x2), dim=1))
        s=self.dec1(torch.cat((self.upconv1(s), x0), dim=1))
        s=self.final_upsample(s)
        s=self.final_conv(s)
        return s

    def forward(self, x):
        # Encoder is shared between classification and segmentation tasks
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.maxpool(x0) 
        x2=self.layer1(x1)  # Output of layer1
        x3=self.layer2(x2)  # Output of layer2
        x4=self.layer3(x3)  # Output of layer3
        x5=self.layer4(x4)  # Output of layer4
        
        # run both headers in parallel for classification and segmentation  
        if not x.is_cuda:
            return self._classification_branch(x5), self._decoder_branch(x5, x4, x3, x2, x0)
        if self.cls_stream is None or self.seg_stream is None:
            self.cls_stream=torch.cuda.Stream(device=x.device)
            self.seg_stream=torch.cuda.Stream(device=x.device)
       
        main_stream=torch.cuda.current_stream(device=x.device)
        self.cls_stream.wait_stream(main_stream)
        self.seg_stream.wait_stream(main_stream)
        # both head run at the same  time on different streams  
        with torch.cuda.stream(self.cls_stream):
            cls_out=self._classification_branch(x5)
        with torch.cuda.stream(self.seg_stream):
            seg_out=self._decoder_branch(x5, x4, x3, x2, x0)
        main_stream.wait_stream(self.cls_stream)
        main_stream.wait_stream(self.seg_stream)
        #tell allocator that the output are consumen on the main stream 
        cls_out.record_stream(main_stream)
        seg_out.record_stream(main_stream)
     
        return cls_out, seg_out


In [102]:
model = MultiTaskResNetUNet(num_classes=4)
x = torch.randn(2, 3, 224, 224)
cls_out, seg_out = model(x)
print("Classification Output Shape:", cls_out.shape) # Expected: (2, 4)
print("Segmentation Output Shape:", seg_out.shape)   # Expected: (2, 1, 112, 112)


Classification Output Shape: torch.Size([2, 4])
Segmentation Output Shape: torch.Size([2, 1, 224, 224])


In [103]:

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1 - dice

class JointLoss(nn.Module):
    def __init__(self, alpha=1.0):
        super(JointLoss, self).__init__()
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.dice_loss = DiceLoss()

    def forward(self, class_logits, class_targets, seg_logits, seg_targets):
        l_class = self.ce_loss(class_logits, class_targets)
        l_seg = self.bce_loss(seg_logits, seg_targets) + self.dice_loss(seg_logits, seg_targets)
        
        total_loss = l_class + self.alpha * l_seg
        return total_loss, l_class, l_seg

In [104]:


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1 - dice

class JointLoss(nn.Module):
    def __init__(self, alpha=1.0):
        super(JointLoss, self).__init__()
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.dice_loss = DiceLoss()

    def forward(self, class_logits, class_targets, seg_logits, seg_targets):
        l_class = self.ce_loss(class_logits, class_targets)
        l_seg = self.bce_loss(seg_logits, seg_targets) + self.dice_loss(seg_logits, seg_targets)
        
        total_loss = l_class + self.alpha * l_seg
        return total_loss, l_class, l_seg

In [105]:
import torch.optim as optim

# 1. Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiTaskResNetUNet(num_classes=4).to(device)

# 2. Define Optimizer and Joint Loss Function
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
criterion = JointLoss(alpha=1.0) # Combines CrossEntropy for classification + BCE/Dice for segmentation

# 3. Training Loop for Axial + Sagittal Data
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    
    for images, labels, masks, _ in dataloader:
        images = images.to(device)
        labels = labels.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        cls_out, seg_out = model(images)
        loss, l_cls, l_seg = criterion(cls_out, labels, seg_out, masks)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    return running_loss / len(dataloader.dataset)

print("Setup complete. Ready to connect BRISC DataLoaders and begin training!")

Setup complete. Ready to connect BRISC DataLoaders and begin training!


In [ ]:
import torch
import torch.optim as optim

# ---------------------------------------------------------
# 1. Device Setup & Model Initialization
# ---------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = MultiTaskResNetUNet(num_classes=4).to(device)

# ---------------------------------------------------------
# 2. Optimizer and Joint Loss Function
# ---------------------------------------------------------
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
criterion = JointLoss(alpha=1.0)

# ---------------------------------------------------------
# 3. Fixed Training Function (Correct Dictionary Unpacking)
# ---------------------------------------------------------
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_cls_loss = 0.0
    running_seg_loss = 0.0
    
    for batch in dataloader:
        images = batch["image"].to(device)
        labels = batch["tumor_label"].to(device)
        masks = batch["mask"].to(device)
        
        optimizer.zero_grad()
        
        cls_out, seg_out = model(images)
        loss, l_cls, l_seg = criterion(cls_out, labels, seg_out, masks)
        
        loss.backward()
        optimizer.step()
        
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        running_cls_loss += l_cls.item() * batch_size
        running_seg_loss += l_seg.item() * batch_size
        
    dataset_size = len(dataloader.dataset)
    return (
        running_loss / dataset_size,
        running_cls_loss / dataset_size,
        running_seg_loss / dataset_size
    )

# ---------------------------------------------------------
# 4. Multi-Epoch Training Execution
# ---------------------------------------------------------
num_epochs = 10
print(f"Starting model training for {num_epochs} epochs on Axial + Sagittal planes...")

for epoch in range(num_epochs):
    epoch_loss, cls_loss, seg_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    print(f"Epoch [{epoch+1}/{num_epochs}] | Total Loss: {epoch_loss:.4f} | Class Loss: {cls_loss:.4f} | Seg Loss: {seg_loss:.4f}")

# ---------------------------------------------------------
# 5. Save Model Checkpoint
# ---------------------------------------------------------
torch.save(model.state_dict(), "multitask_resnet_run2_axial_sagittal.pth")
print("Training complete! Model saved as 'multitask_resnet_run2_axial_sagittal.pth'.")

Using device: cpu
Starting model training for 10 epochs on Axial + Sagittal planes...
